In [1]:
!pip install pyspark -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving customers.csv to customers.csv
Saving Data_Processing_.ipynb to Data_Processing_.ipynb
Saving orders_cleaned.csv to orders_cleaned.csv


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count as spark_count

spark = SparkSession.builder \
    .appName("CustomerOrderInsights") \
    .getOrCreate()

In [4]:
orders_df = spark.read.csv("orders_cleaned.csv", header=True, inferSchema=True)
orders_df = orders_df.drop("region")

customers_df = spark.read.csv("customers.csv", header=True, inferSchema=True)

orders_df.printSchema()
customers_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- issue: string (nullable = true)
 |-- is_delivered: boolean (nullable = true)
 |-- delay_days: integer (nullable = true)
 |-- delayed: integer (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- region: string (nullable = true)



In [5]:
joined_df = orders_df.join(customers_df, on="customer_id", how="inner")
joined_df.select("order_id", "customer_id", "name", "region", "delayed", "delay_days").show(10)

+--------+-----------+--------------+------+-------+----------+
|order_id|customer_id|          name|region|delayed|delay_days|
+--------+-----------+--------------+------+-------+----------+
|       1|          1|    Asha Patel|  West|      1|         2|
|       2|          2|    Ravi Kumar| North|      0|         0|
|       3|          3|    Meena Iyer| South|      1|         2|
|       4|          1|    Asha Patel|  West|      1|         2|
|       5|          4|   John Carter|  East|      1|         6|
|       6|          5|    Priya Nair| South|      0|         0|
|       7|          6|  Vikram Singh| North|      1|         1|
|       8|          7|   Sara Thomas|  West|      0|         0|
|       9|          8|   Arjun Mehta|  East|      0|         0|
|      10|          9|Lena Fernandes| South|      0|         0|
+--------+-----------+--------------+------+-------+----------+
only showing top 10 rows


In [6]:
region_summary = (
    joined_df.groupBy("region")
    .agg(
        spark_count("order_id").alias("total_orders"),
        spark_sum("delayed").alias("delayed_orders"),
        spark_sum("delay_days").alias("total_delay_days")
    )
    .orderBy(col("delayed_orders").desc())
)
region_summary.show()

+------+------------+--------------+----------------+
|region|total_orders|delayed_orders|total_delay_days|
+------+------------+--------------+----------------+
|  West|           8|             4|               7|
| South|           7|             2|               4|
|  East|           4|             2|               7|
| North|           5|             1|               1|
+------+------------+--------------+----------------+



In [7]:
region_summary.coalesce(1).write.mode("overwrite").option("header", True).csv("delayed_orders_by_region")

import glob, shutil
part_file = glob.glob("delayed_orders_by_region/part-*.csv")[0]
shutil.copy(part_file, "delayed_orders_by_region.csv")

files.download("delayed_orders_by_region.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>